In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import joblib  # This is used to save the model for the app

In [ ]:
# Create the training data (features and target)
data = {
    'hrv': [80, 40, 60, 30, 90, 25, 70, 45, 50, 35],
    'sleep': [8, 5, 7, 4, 9, 3, 7.5, 6, 6.5, 5],
    'intensity': [3, 9, 5, 8, 2, 10, 4, 7, 6, 9],
    'prev_injury': [0, 1, 0, 1, 0, 1, 0, 0, 1, 1],
    'injury_risk': [5, 85, 20, 95, 2, 99, 15, 60, 55, 88] # Target Injury %
}

df = pd.DataFrame(data)

# Split into X (inputs) and y (what we want to predict)
X = df[['hrv', 'sleep', 'intensity', 'prev_injury']]
y = df['injury_risk']

# Train the AI Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

print("✅ Step 3 Complete: The AI has learned the patterns!")

✅ Step 3 Complete: The AI has learned the patterns!


In [ ]:
# 1. Test a prediction (Example: Low HRV, bad sleep, high intensity)
test_user = [[30, 4, 9, 1]]
prediction = model.predict(test_user)
print(f"Predicted Injury Risk for demo: {prediction[0]:.2f}%")

# 2. Save the model to use in your app
joblib.dump(model, 'injury_model.pkl')
print("✅ Step 4 Complete: 'injury_model.pkl' is ready for download!")

Predicted Injury Risk for demo: 93.76%
✅ Step 4 Complete: 'injury_model.pkl' is ready for download!


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [ ]:
# @title  Guardian Fit
# @markdown Enter the athlete's stats below to calculate risk:

HRV = 34 # @param {type:"slider", min:10, max:100, step:1}
Sleep_Hours = 4 # @param {type:"slider", min:1, max:12, step:1}
Intensity = 8 # @param {type:"slider", min:1, max:10, step:1}
Previous_Injury = "No" # @param ["No", "Yes"]

# Convert Yes/No to 1/0
prev_inj_val = 1 if Previous_Injury == "Yes" else 0

# Use the model you already trained (Aaryan's model)
prediction = model.predict([[HRV, Sleep_Hours, Intensity, prev_inj_val]])
risk_score = prediction[0]


print(f"PREDICTED INJURY RISK: {risk_score:.2f}%")

if risk_score > 70:
    print("RECOMMENDATION: HIGH RISK. Mandatory Rest Day.")
elif risk_score > 30:
    print("RECOMMENDATION: MODERATE RISK. Light Mobility/Stretching Only.")
else:
    print("RECOMMENDATION: LOW RISK. Cleared for High-Intensity Training!")

PREDICTED INJURY RISK: 83.05%
RECOMMENDATION: HIGH RISK. Mandatory Rest Day.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [ ]:
!pip install gradio -q

import gradio as gr

# The prediction function for the interface
def predict_injury(hrv, sleep, intensity, prev_injury):
    # Convert "Yes"/"No" to 1/0
    inj_val = 1 if prev_injury == "Yes" else 0

    # Use your trained model
    pred = model.predict([[hrv, sleep, intensity, inj_val]])
    risk = pred[0]

    # Determine the status and advice
    if risk > 70:
        return f"{risk:.1f}%", "HIGH RISK: Mandatory Rest Day."
    elif risk > 30:
        return f"{risk:.1f}%", "MODERATE RISK: Light stretching only."
    else:
        return f"{risk:.1f}%", "LOW RISK: Cleared for training!"

# Create the UI
with gr.Blocks(title="Guardian Fit") as demo:
    gr.Markdown("# Guardian Fit")
    gr.Markdown("### Developed by Aaryan, Rajesh, & Umesh")

    with gr.Row():
        with gr.Column():
            hrv_in = gr.Slider(10, 100, value=50, label="Heart Rate Variability (HRV)")
            sleep_in = gr.Slider(1, 12, value=7, label="Hours of Sleep")
            intensity_in = gr.Slider(1, 10, value=5, label="Workout Intensity")
            injury_in = gr.Radio(["No", "Yes"], value="No", label="Previous Injury?")
            btn = gr.Button("Calculate Recovery Score")

        with gr.Column():
            risk_out = gr.Label(label="Predicted Injury Risk")
            advice_out = gr.Textbox(label="AI Recommendation")

    btn.click(predict_injury, inputs=[hrv_in, sleep_in, intensity_in, injury_in], outputs=[risk_out, advice_out])

# This creates a public link for your presentation!
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1d6775a6923cbb4908.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install gradio -q
import gradio as gr

# 1. AI Logic Function
def predict_injury(hrv, sleep, intensity, prev_injury):
    try:
        # Convert "Yes"/"No" to 1/0
        inj_val = 1 if prev_injury == "Yes" else 0

        # Ensure input is in the correct shape for the AI
        input_data = [[float(hrv), float(sleep), float(intensity), inj_val]]

        # Use the model to predict
        pred = model.predict(input_data)
        risk = pred[0]

        if risk > 70:
            return f"{risk:.1f}%", "HIGH RISK: Mandatory Rest Day."
        elif risk > 30:
            return f"{risk:.1f}%", "MODERATE RISK: Light recovery only."
        else:
            return f"{risk:.1f}%", "LOW RISK: Cleared for training!"

    except Exception as e:
        return "Error", f"AI Brain not found. Please re-run the Training Cell. (Detail: {e})"

# 2. Custom Professional Styling (CSS)
custom_css = """
body { background-image: linear-gradient(135deg, #0f0c29, #302b63, #24243e); color: white; }
.gradio-container { background: rgba(255, 255, 255, 0.05) !important; backdrop-filter: blur(10px); border-radius: 20px; border: 1px solid rgba(255, 255, 255, 0.1); padding: 20px; }
h1 { color: #00d2ff !important; text-align: center; font-family: 'Inter', sans-serif; text-transform: uppercase; letter-spacing: 2px; }
.gr-button-primary { background: linear-gradient(90deg, #00d2ff, #3a7bd5) !important; border: none !important; font-weight: bold !important; }
label { color: #adb5bd !important; font-weight: 600 !important; }
"""

# 3. Building the Interface
with gr.Blocks(css=custom_css, title="Guardian Fit") as demo:
    gr.Markdown("# Guardian Fit")
    gr.Markdown("<p style='text-align: center; color: #adb5bd;'>Injury Risk Predictor  </p>")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Athlete Inputs")
            hrv_in = gr.Slider(10, 100, value=50, label="HRV (ms)")
            sleep_in = gr.Slider(1, 12, value=7, label="Sleep (Hours)")
            intensity_in = gr.Slider(1, 10, value=5, label="Session Intensity")
            injury_in = gr.Radio(["No", "Yes"], value="No", label="Previous Injury History")
            btn = gr.Button("ANALYZE READINESS", variant="primary")

        with gr.Column():
            gr.Markdown("### AI Diagnostics")
            risk_out = gr.Label(label="Calculated Injury Risk")
            advice_out = gr.Textbox(label="Smart Recommendation", interactive=False)

    btn.click(predict_injury, inputs=[hrv_in, sleep_in, intensity_in, injury_in], outputs=[risk_out, advice_out])

# Launch with share=True to get a public link for the judges
demo.launch(share=True)

/tmp/ipykernel_8479/3319310108.py:37: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="Guardian Fit") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c8c4f3174dcff4f277.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
